---
### **Factor construction (ivol 전략)**

**Idiosyncratic Volatility (ivol) 계산 방법**

- **모형**: Carhart 4-factor  
- **데이터**: 각 종목의 월별 초과수익률  
- **방법**: 60개월 이동 윈도우(rolling window)로 회귀 분석  
- **회귀식**:

$$
R_{i,t} - R_{f,t} 
= \alpha_i 
+ \beta_{MKT}(MKT_t) 
+ \beta_{SMB}(SMB_t) 
+ \beta_{HML}(HML_t) 
+ \beta_{MOM}(MOM_t) 
+ \epsilon_{i,t}
$$

- $ \epsilon_{i,t} $ : 잔차, 개별 종목의 고유 수익률 (idiosyncratic return)


- **ivol 정의**: 각 윈도우별 잔차 $ \epsilon_{i,t} $ 의 표준편차  

$$
ivol_{i,t} = \operatorname{std}\left(\epsilon_{i, t-59:t}\right)
$$

- **전략 적용**: 저 ivol(저변동성) 종목에 롱 포지션, 고 ivol(고변동성) 종목에 숏 포지션

---
**데이터 로드**

In [48]:
import pandas as pd

# Investment Factor 계산에 필요한 데이터를 input 폴더에서 불러옴
excess_return    = pd.read_csv("./input/excess_return.csv", index_col=0, parse_dates=True)     # 초과수익률 (종속변수)
factors          = pd.read_csv("./input/factors_monthly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)

---
**독립변수 설정**

In [49]:
# 수익률 분해를 위한 팩터 정리 - 독립변수 설정
factors['MKT'] = factors['KOSPI'] - factors['RF']            # MKT 팩터(시장 포트폴리오 초과 수익률)
X = factors[['MKT', 'SMB', 'HML', 'MOM']].copy()

In [50]:
X.tail()

,MKT,SMB,HML,MOM
Date,,,,
2025-04-30,0.028195,0.054326,-0.011127,0.054588
2025-05-31,0.053041,-0.017765,0.061506,0.047865
2025-06-30,0.136541,-0.067961,0.028903,0.011075
2025-07-31,0.054494,-0.043851,-0.025323,-0.064316
2025-08-31,-0.020396,-0.002987,-0.001854,-0.008658


---
**종속변수 설정**

In [52]:
excess_return.tail()

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,HD현대중공업,기아,KB금융,두산에너빌리티,...,웨이포트,성융광전투자,완리,골든센츄리,평산차업 KDR,네프로아이티,중국고섬,SBI모기지,SBI핀테크솔루션즈,SNK
Date,,,,,,,,,,,,,,,,,,,,,
2025-04-30,-0.042023,-0.071449,-0.032126,0.035245,0.268901,-0.035699,0.438629,-0.022816,0.139541,0.232311,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.005280,NaN
2025-05-31,0.010479,0.149980,-0.120777,-0.021145,0.031237,-0.029940,0.006573,-0.013195,0.154186,0.391649,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.002133,NaN
2025-06-30,0.061948,0.425764,0.036353,-0.040868,0.043514,0.096110,0.054611,0.081784,0.061170,0.693059,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002988,NaN
2025-07-31,0.191912,-0.065424,0.285811,0.073537,0.172460,0.044615,0.142623,0.053660,-0.002068,-0.043004,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-08-31,-0.025894,-0.018538,-0.081823,-0.063940,-0.114534,0.030780,0.058058,0.032129,-0.026431,-0.061535,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
**ivol 계산 함수 정의**

- **방법**: 60개월 이동 윈도우(rolling window)로 회귀 분석  
- **회귀식**:

$$
R_{i,t} - R_{f,t} 
= \alpha_i 
+ \beta_{MKT}(MKT_t) 
+ \beta_{SMB}(SMB_t) 
+ \beta_{HML}(HML_t) 
+ \beta_{MOM}(MOM_t) 
+ \epsilon_{i,t}
$$

- $ \epsilon_{i,t} $ : 잔차, 개별 종목의 고유 수익률 (idiosyncratic return)


- **ivol 정의**: 각 윈도우별 잔차 $ \epsilon_{i,t} $ 의 표준편차

$$
ivol_{i,t} = \operatorname{std}\left(\epsilon_{i, t-59:t}\right)
$$

In [53]:
import statsmodels.api as sm
import numpy as np

# iVOL 계산 함수 정의
def rolling_ivol(y: pd.Series,
                 X: pd.DataFrame,
                 window: int = 60) -> pd.Series:
    """
    개별 종목의 idiosyncratic volatility (iVOL) 시계열을 계산.
    
    Parameters
    ----------
    y : pd.Series
        종목의 초과수익률 (index = datetime).
    X : pd.DataFrame
        팩터 수익률 (MKT, SMB, HML, MOM).
    window : int, default 60
        롤링 회귀 구간(개월 수).
    
    Returns
    -------
    pd.Series
        각 시점별 iVOL (60개월 잔차의 표준편차).
    """
    df = pd.concat([y.rename("y"), X], axis=1).dropna()
    ivol = pd.Series(index=y.index, dtype=float)

    for t in range(len(df)):
        end_idx = df.index[t]
        start_pos = max(0, t - window + 1)
        train = df.iloc[start_pos:t+1]

        if len(train) < window:
            continue

        # 회귀
        y_win = train['y']
        X_win = sm.add_constant(train[['MKT','SMB','HML','MOM']], has_constant='add')
        model = sm.OLS(y_win.values, X_win.values).fit()

        # 해당 윈도우 잔차의 표준편차 = iVOL
        resid_std = np.std(model.resid, ddof=1)
        ivol.loc[end_idx] = resid_std

    return ivol

---
**ivol 계산(병렬 처리를 이용)**

In [54]:
from joblib import Parallel, delayed
from tqdm import tqdm

tickers = excess_return.columns.tolist()

# 병렬로 iVOL 계산
ivol_list = Parallel(n_jobs=-1)(
    delayed(rolling_ivol)(excess_return[ticker], X, window=60)
    for ticker in tqdm(tickers, desc="Calculating iVOL")
)

ivol_df = pd.concat(ivol_list, axis=1)
ivol_df.columns = tickers

Calculating iVOL: 100%|████████████████████████████████████████████████████████████| 3851/3851 [02:43<00:00, 23.51it/s]


In [56]:
ivol_df.tail()

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,HD현대중공업,기아,KB금융,두산에너빌리티,...,웨이포트,성융광전투자,완리,골든센츄리,평산차업 KDR,네프로아이티,중국고섬,SBI모기지,SBI핀테크솔루션즈,SNK
Date,,,,,,,,,,,,,,,,,,,,,
2025-04-30,0.046126,0.077278,NaN,0.067957,0.125740,0.084412,NaN,0.077852,0.071672,0.189411,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.116061,NaN
2025-05-31,0.046472,0.077139,NaN,0.067908,0.123177,0.086175,NaN,0.077817,0.071851,0.191410,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.115267,NaN
2025-06-30,0.047999,0.081132,NaN,0.065427,0.123540,0.086487,NaN,0.078068,0.072243,0.199805,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.114454,NaN
2025-07-31,0.049854,0.082132,NaN,0.064716,0.122314,0.081311,NaN,0.074403,0.072121,0.155505,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-08-31,0.048234,0.079448,NaN,0.065198,0.123846,0.066232,NaN,0.074579,0.072118,0.143976,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
**데이터 저장**

In [57]:
# Output 폴더
ivol_df.to_csv("./output/ivol.csv", index=True, encoding="utf-8-sig")

# 데이터가 필요한 단계의 input 폴더
ivol_df.to_csv("../Step3 포트폴리오 구성 (Portfolio formation)/input/ivol.csv", index=True, encoding="utf-8-sig")